In [0]:
# ============================================
# Bronze Layer - NYC Taxi Data Ingestion
# ============================================
# ✅ 從 Key Vault 攞 key，唔再 hardcode！
storage_key = dbutils.secrets.get(
    scope="kv-dataeng-scope",
    key="storage-account-key"
)

spark.conf.set(
    "fs.azure.account.key.stadataengdev.dfs.core.windows.net",
    storage_key
)

# Storage Account 設定
storage_account_name = "stadataengdev"
container_name = "raw"
file_name = "yellow_tripdata_2024-01.parquet"

# 讀取數據 (暫時用 Access Key，之後換 Key Vault)
df = spark.read.parquet(
    f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{file_name}"
)

# 睇下數據
print(f"總行數: {df.count():,}")
print(f"總列數: {len(df.columns)}")
df.printSchema()

總行數: 2,964,624
總列數: 19
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [0]:
# 睇頭5行數據
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [0]:
# 基本統計
print("=== 數據質量檢查 ===")

# 空值檢查
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])
null_counts.show()

=== 數據質量檢查 ===
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         140162|            0|    140162|            140162|           0|           0|           0|          0|    0|  

In [0]:
# ============================================
# Silver Layer - Data Cleaning
# ============================================
from pyspark.sql.functions import col, when, current_timestamp

# 清洗規則：
# 1. 刪除 fare_amount <= 0
# 2. 刪除 passenger_count 空值或 <= 0
# 3. 刪除 trip_distance <= 0

df_silver = df.filter(
    (col("fare_amount") > 0) &
    (col("passenger_count") > 0) &
    (col("passenger_count").isNotNull()) &
    (col("trip_distance") > 0)
).withColumn(
    "ingestion_timestamp", current_timestamp()
)

print(f"清洗前行數: {df.count():,}")
print(f"清洗後行數: {df_silver.count():,}")
print(f"刪除行數: {df.count() - df_silver.count():,}")

清洗前行數: 2,964,624
清洗後行數: 2,723,805
刪除行數: 240,819


In [0]:
# ============================================
# 存去 Silver Layer (processed/)
# ============================================

silver_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/nyc_taxi/2024/01/"

df_silver.write \
    .mode("overwrite") \
    .parquet(silver_path)

print(f"✅ Silver Layer 存檔完成！")
print(f"路徑: {silver_path}")
print(f"總行數: {df_silver.count():,}")

✅ Silver Layer 存檔完成！
路徑: abfss://processed@stadataengdev.dfs.core.windows.net/nyc_taxi/2024/01/
總行數: 2,723,805


In [0]:
# ============================================
# Gold Layer - Aggregation
# ============================================
from pyspark.sql.functions import avg, count, round as spark_round, hour

# 計算每小時平均車費同行程數
df_gold = df_silver.withColumn(
    "pickup_hour", hour(col("tpep_pickup_datetime"))
).groupBy("pickup_hour") \
.agg(
    count("*").alias("total_trips"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("tip_amount"), 2).alias("avg_tip")
).orderBy("pickup_hour")

print("=== 每小時統計 ===")
df_gold.show(24)

=== 每小時統計 ===
+-----------+-----------+--------+------------+-------+
|pickup_hour|total_trips|avg_fare|avg_distance|avg_tip|
+-----------+-----------+--------+------------+-------+
|          0|      69189|   19.71|        3.89|   3.61|
|          1|      45376|   17.36|        3.25|   3.27|
|          2|      31932|   16.36|        3.02|   3.07|
|          3|      20549|   17.88|        3.46|   3.13|
|          4|      12783|   23.01|        4.79|   3.57|
|          5|      15629|   27.58|        6.21|   4.09|
|          6|      35583|   21.99|        4.72|   3.35|
|          7|      74314|   18.55|        3.54|   3.21|
|          8|     105119|   17.59|        3.06|   3.18|
|          9|     119392|   17.86|        3.04|   3.24|
|         10|     130036|   18.05|        3.05|   3.22|
|         11|     141260|    17.6|        2.98|   3.18|
|         12|     153606|   17.78|        2.98|   3.21|
|         13|     159088|   18.39|        3.16|   3.32|
|         14|     171274|   19.24|

In [0]:
# ============================================
# 存去 Gold Layer (curated/) - Delta Format
# ============================================

gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/nyc_taxi_hourly_stats/"

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .save(gold_path)

print(f"✅ Gold Layer 存檔完成！")
print(f"路徑: {gold_path}")
print(f"Format: Delta Lake")

✅ Gold Layer 存檔完成！
路徑: abfss://curated@stadataengdev.dfs.core.windows.net/nyc_taxi_hourly_stats/
Format: Delta Lake
